In [1]:
!pip install xgboost


In [2]:
#Basic Libraries (Data Handling)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


#Visualization Libraries
import matplotlib.pyplot as plt
import seaborn as sns

#Preprocessing & Utilities
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer

#Machine Learning Models
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from sklearn.neural_network import MLPClassifier

#Model Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score
)
#save model
import pickle
import joblib
import os


In [3]:
#Load Dataset
df = pd.read_csv(r"C:\Users\chait\OneDrive\Desktop\Phishing Desing\phishing_dataset.csv")
print(df.head())

                                                 url  length_url  \
0              http://www.crestonwood.com/router.php          37   
1  http://shadetreetechnology.com/V4/validation/a...          77   
2  https://support-appleld.com.secureupdate.duila...         126   
3                                 http://rgipt.ac.in          18   
4  http://www.iracing.com/tracks/gateway-motorspo...          55   

   length_hostname  ip  nb_dots  nb_hyphens  nb_at  nb_qm  nb_and  nb_or  ...  \
0               19   0        3           0      0      0       0      0  ...   
1               23   1        1           0      0      0       0      0  ...   
2               50   1        4           1      0      1       2      0  ...   
3               11   0        2           0      0      0       0      0  ...   
4               15   0        2           2      0      0       0      0  ...   

   domain_in_title  domain_with_copyright  whois_registered_domain  \
0                0                

### Data Understanding

In [4]:
print("Shape of dataset:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

Shape of dataset: (11430, 89)

Columns:
 ['url', 'length_url', 'length_hostname', 'ip', 'nb_dots', 'nb_hyphens', 'nb_at', 'nb_qm', 'nb_and', 'nb_or', 'nb_eq', 'nb_underscore', 'nb_tilde', 'nb_percent', 'nb_slash', 'nb_star', 'nb_colon', 'nb_comma', 'nb_semicolumn', 'nb_dollar', 'nb_space', 'nb_www', 'nb_com', 'nb_dslash', 'http_in_path', 'https_token', 'ratio_digits_url', 'ratio_digits_host', 'punycode', 'port', 'tld_in_path', 'tld_in_subdomain', 'abnormal_subdomain', 'nb_subdomains', 'prefix_suffix', 'random_domain', 'shortening_service', 'path_extension', 'nb_redirection', 'nb_external_redirection', 'length_words_raw', 'char_repeat', 'shortest_words_raw', 'shortest_word_host', 'shortest_word_path', 'longest_words_raw', 'longest_word_host', 'longest_word_path', 'avg_words_raw', 'avg_word_host', 'avg_word_path', 'phish_hints', 'domain_in_brand', 'brand_in_subdomain', 'brand_in_path', 'suspecious_tld', 'statistical_report', 'nb_hyperlinks', 'ratio_intHyperlinks', 'ratio_extHyperlinks', 

In [5]:
#Encode target column

print(df["status"].value_counts())

status
legitimate    5715
phishing      5715
Name: count, dtype: int64


In [6]:
#Convert target labels into numbers:

df["status"] = df["status"].map({
    "legitimate": 0,
    "phishing": 1
})

In [7]:
#Drop unwanted column

if "url" in df.columns:
    df = df.drop("url", axis=1)

In [8]:
#Separate features and target
X = df.drop("status", axis=1)
y = df["status"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (11430, 87)
y shape: (11430,)


In [9]:
#Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

X_train shape: (9144, 87)
X_test shape : (2286, 87)


### Create preprocessing pipeline

In [10]:
#Create preprocessing pipeline
#Since your data is mostly numeric, use:

SimpleImputer(strategy="median")
StandardScaler()
numeric_features = X.columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features)
    ]
)

In [11]:
def evaluate_model(model_name, y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)

    print(f"\n{'='*50}")
    print(f"Model: {model_name}")
    print(f"{'='*50}")
    print("Accuracy :", round(acc, 4))
    print("Precision:", round(pre, 4))
    print("Recall   :", round(rec, 4))
    print("F1 Score :", round(f1, 4))
    print("ROC-AUC  :", round(auc, 4))

    print("\nClassification Report:\n")
    print(classification_report(y_true, y_pred))

    print("Confusion Matrix:\n")
    print(confusion_matrix(y_true, y_pred))

    return {
        "Model": model_name,
        "Accuracy": acc,
        "Precision": pre,
        "Recall": rec,
        "F1 Score": f1,
        "ROC-AUC": auc
    }

In [12]:
#Random Forest pipeline
rf_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=200,
        max_depth=15,
        random_state=42,
        class_weight="balanced"
    ))
])

rf_pipeline.fit(X_train, y_train)

rf_pred = rf_pipeline.predict(X_test)
rf_prob = rf_pipeline.predict_proba(X_test)[:, 1]

rf_results = evaluate_model("Random Forest Pipeline", y_test, rf_pred, rf_prob)


Model: Random Forest Pipeline
Accuracy : 0.9624
Precision: 0.96
Recall   : 0.965
F1 Score : 0.9625
ROC-AUC  : 0.993

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.96      0.96      1143
           1       0.96      0.97      0.96      1143

    accuracy                           0.96      2286
   macro avg       0.96      0.96      0.96      2286
weighted avg       0.96      0.96      0.96      2286

Confusion Matrix:

[[1097   46]
 [  40 1103]]


In [13]:
#XGBoost pipeline
xgb_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", XGBClassifier(
        n_estimators=200,
        max_depth=8,
        learning_rate=0.1,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric="logloss"
    ))
])

xgb_pipeline.fit(X_train, y_train)

xgb_pred = xgb_pipeline.predict(X_test)
xgb_prob = xgb_pipeline.predict_proba(X_test)[:, 1]

xgb_results = evaluate_model("XGBoost Pipeline", y_test, xgb_pred, xgb_prob)


Model: XGBoost Pipeline
Accuracy : 0.9663
Precision: 0.9603
Recall   : 0.9729
F1 Score : 0.9665
ROC-AUC  : 0.9943

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.96      0.97      1143
           1       0.96      0.97      0.97      1143

    accuracy                           0.97      2286
   macro avg       0.97      0.97      0.97      2286
weighted avg       0.97      0.97      0.97      2286

Confusion Matrix:

[[1097   46]
 [  31 1112]]


In [14]:
#Neural Network pipeline
mlp_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", MLPClassifier(
        hidden_layer_sizes=(128, 64),
        activation="relu",
        solver="adam",
        max_iter=300,
        random_state=42
    ))
])

mlp_pipeline.fit(X_train, y_train)

mlp_pred = mlp_pipeline.predict(X_test)
mlp_prob = mlp_pipeline.predict_proba(X_test)[:, 1]

mlp_results = evaluate_model("MLP Pipeline", y_test, mlp_pred, mlp_prob)


Model: MLP Pipeline
Accuracy : 0.9567
Precision: 0.9555
Recall   : 0.958
F1 Score : 0.9567
ROC-AUC  : 0.9889

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.96      0.96      1143
           1       0.96      0.96      0.96      1143

    accuracy                           0.96      2286
   macro avg       0.96      0.96      0.96      2286
weighted avg       0.96      0.96      0.96      2286

Confusion Matrix:

[[1092   51]
 [  48 1095]]


In [15]:
#Ensemble Voting pipeline

#This is your final advanced model.

ensemble_model = VotingClassifier(
    estimators=[
        ("rf", rf_pipeline),
        ("xgb", xgb_pipeline),
        ("mlp", mlp_pipeline)
    ],
    voting="soft"
)

ensemble_model.fit(X_train, y_train)

ensemble_pred = ensemble_model.predict(X_test)
ensemble_prob = ensemble_model.predict_proba(X_test)[:, 1]

ensemble_results = evaluate_model("Ensemble Voting Pipeline", y_test, ensemble_pred, ensemble_prob)


Model: Ensemble Voting Pipeline
Accuracy : 0.9681
Precision: 0.9668
Recall   : 0.9694
F1 Score : 0.9681
ROC-AUC  : 0.9939

Classification Report:

              precision    recall  f1-score   support

           0       0.97      0.97      0.97      1143
           1       0.97      0.97      0.97      1143

    accuracy                           0.97      2286
   macro avg       0.97      0.97      0.97      2286
weighted avg       0.97      0.97      0.97      2286

Confusion Matrix:

[[1105   38]
 [  35 1108]]


In [16]:
#Compare all models
results_df = pd.DataFrame([
    rf_results,
    xgb_results,
    mlp_results,
    ensemble_results
])

results_df.sort_values(by="Accuracy", ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score,ROC-AUC
3,Ensemble Voting Pipeline,0.968066,0.966841,0.969379,0.968108,0.993866
1,XGBoost Pipeline,0.966317,0.960276,0.972878,0.966536,0.994326
0,Random Forest Pipeline,0.962380,0.959965,0.965004,0.962478,0.993015
2,MLP Pipeline,0.956693,0.955497,0.958005,0.956750,0.988859


In [17]:
os.makedirs("model", exist_ok=True)

In [18]:
joblib.dump(ensemble_model, "model/ensemble_pipeline_model.pkl")
print("Ensemble pipeline model saved successfully.")

Ensemble pipeline model saved successfully.


In [19]:
loaded_model = joblib.load("model/ensemble_pipeline_model.pkl")
print("Model loaded successfully.")

Model loaded successfully.
